### DS4420 Project Phase 1

Authors: Gavin Bond, Benjamin Kataoka, Preetish Paul

## Proof of Concept Model:

Our proof of concept model will be a Neural Network, exploring features that affect Developer's trust in AI output. We will start with the 2025 dataset only for simplicity. If you need to rerun notebook, instructions to access the dataset are in the "README.md" file; GitHub did not allow us to upload the data to the repository due to the dataset size exceeding their limit, so it must be downloaded locally for now. We will find a better solution for the final project, but thought this would suffice for now.

In [2]:
# Necessary Imports:
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Dense
from keras.optimizers import SGD
from keras.losses import binary_crossentropy

In [3]:
# File Location
file_2025 = "data/stack-overflow-developer-survey-2025/survey_results_public.csv"

# Import to pandas df
df = pd.read_csv(file_2025)

# View df head
print(df.columns.tolist())
print(df.shape)

/var/folders/_6/ywqm7cjj7n3bjzpgtsy8c0vc0000gn/T/ipykernel_92670/3342372067.py:5: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_2025)


['ResponseId', 'MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl', 'WorkExp', 'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'YearsCode', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'PurchaseInfluence', 'TechEndorseIntro', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13', 'TechEndorse_13_TEXT', 'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5', 'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13', 'TechOppose_16', 'TechOppose_15', 'TechOppose_15_TEXT', 'Industry', 'JobSatPoints_1', 'JobSatPoints_2', 'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6', 'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10', 'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14', 'JobSatPoints_15', 'JobSatPoints_16', 'JobSatPoints_15_TEXT', 'AIThreat', 'NewRole', 'ToolCountWork', 'ToolCountPersona

There are clearly a litany of columns and potential features. While we will implement more extensive feature engineering for the final version, we have compiled a solid starting set upon reading the survey & schema itself: 

Target:
- "AIAcc": How much do you trust the accuracy of the output from AI tools as part of your development workflow?

Features:
- "Age": Surveyee age
- "AIThreat": Do you believe AI is a threat to your current job?
- "YearsCode": How many years surveyee has spend coding (educational & professional)
- "RemoteWork": Developers WFH or office prescence

While not a feature, we will first filter our data to only include Developers. From the survey options, we can see this option is "I am a developer by profession".

In [5]:
# Professional developer filter
df_dev = df[df["MainBranch"] == "I am a developer by profession"].copy()

print(df_dev.shape)

(37467, 172)


This whittled down our dataset size slightly; next, we will define feature and target columns, and drop the rest.

In [7]:
# Target & feature columns
targ = "AIAcc"
feats = ["Age", "AIThreat", "YearsCode", "RemoteWork"]

# DF copy with only targ and features
df_model = df_dev[[targ] + feats].copy()
df_model.head()

,AIAcc,Age,AIThreat,YearsCode,RemoteWork
0,Neither trust nor distrust,25-34 years old,I'm not sure,14.0,Remote
1,Neither trust nor distrust,25-34 years old,I'm not sure,10.0,"Hybrid (some in-person, leans heavy to flexibi..."
2,Somewhat trust,35-44 years old,No,12.0,NaN
3,Somewhat trust,35-44 years old,No,5.0,Remote
4,Neither trust nor distrust,35-44 years old,No,22.0,NaN


Next, we will drop NA values for the target variable. and view the new dataset size.

In [9]:
# Drop NA for targ
df_model = df_model[df_model[targ].notna()].copy()
print(df_model.shape)
df_model.head()

(25739, 5)


,AIAcc,Age,AIThreat,YearsCode,RemoteWork
0,Neither trust nor distrust,25-34 years old,I'm not sure,14.0,Remote
1,Neither trust nor distrust,25-34 years old,I'm not sure,10.0,"Hybrid (some in-person, leans heavy to flexibi..."
2,Somewhat trust,35-44 years old,No,12.0,NaN
3,Somewhat trust,35-44 years old,No,5.0,Remote
4,Neither trust nor distrust,35-44 years old,No,22.0,NaN


Next we will make a binary target variable for a simple NN approach. "Somewhat distrust" or "highly distrust" are modeled as distrust, everything else as trust.

In [11]:
# Binary encoding trust target variable
df_model["y"] = df_model[targ].astype(str).str.contains(
    "Somewhat distrust|Highly distrust",
    case=False,
    na=False
).astype(int)

# Confirming binary target produced sufficient "distrust" outputs for modeling & comparing to "trust" total
print((df_model["y"] == 1).sum())
print((df_model["y"] == 0).sum())
df_model.head()

11825
13914


,AIAcc,Age,AIThreat,YearsCode,RemoteWork,y
0,Neither trust nor distrust,25-34 years old,I'm not sure,14.0,Remote,0
1,Neither trust nor distrust,25-34 years old,I'm not sure,10.0,"Hybrid (some in-person, leans heavy to flexibi...",0
2,Somewhat trust,35-44 years old,No,12.0,NaN,0
3,Somewhat trust,35-44 years old,No,5.0,Remote,0
4,Neither trust nor distrust,35-44 years old,No,22.0,NaN,0


Next, we will clean the data simply: numerical columns filled with the median value, categorical cols filled with "NA" str.

In [13]:
# identify column type for clean method
categorical_cols = ["Age", "AIThreat", "RemoteWork"]
numeric_cols = ["YearsCode"]

# Fill with median
for col in numeric_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

# Fill with "NA" string
for col in categorical_cols:
    df_model[col] = df_model[col].fillna("NA").astype(str)

# Check for remaining NA values
total_nas = df_model.isna().sum().sum()
print(total_nas)

0


Now, we will build our X and y series, with a one-hot encoder for the categorical columns

In [15]:
# Features
X = df_model[["Age", "AIThreat", "YearsCode", "RemoteWork"]].copy()
# Target variable
y = df_model["y"].copy()

# Encode categorical columns
X_encoded = pd.get_dummies(X, columns=["Age", "AIThreat", "RemoteWork"], drop_first=True)

Now, we can split into training and testing sets

In [17]:
# T/T split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

Now, we can build out our Neural Network with the Keras library explored in lecture 3. We will keep the architecture simple:
- Input layer
- 2 hidden layers, both relu activation
- Output layer with sigmoid activation for binary classification

In [19]:
# Input layer
input = Input(shape=(X_train.shape[1],))

# Hidden layers, both relu
hidden1 = Dense(20, activation="relu")(input)
hidden2 = Dense(10, activation="relu")(hidden1)

# Output layer with sigmoid
output = Dense(1, activation="sigmoid")(hidden2)

# Building model with above layers
model = Model(inputs=input, outputs=output)

# Compiling model
model.compile(optimizer=SGD(),
              loss=binary_crossentropy,
              metrics=['accuracy'])

Now, let's fit on the training set

In [21]:
# Fit model (10 epochs)
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 1s 964us/step - accuracy: 0.5216 - loss: 0.7131
Epoch 2/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 664us/step - accuracy: 0.5472 - loss: 0.6895
Epoch 3/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 709us/step - accuracy: 0.5517 - loss: 0.6862
Epoch 4/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 570us/step - accuracy: 0.5531 - loss: 0.6837
Epoch 5/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step - accuracy: 0.5557 - loss: 0.6833
Epoch 6/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 528us/step - accuracy: 0.5582 - loss: 0.6822
Epoch 7/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 567us/step - accuracy: 0.5616 - loss: 0.6813
Epoch 8/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 630us/step - accuracy: 0.5572 - loss: 0.6814
Epoch 9/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 1s 785us/step - accuracy: 0.5604 - loss: 0.6809
Epoch 10/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 1s 921us/step - accuracy: 0.5616 - loss: 0.6805


Now, let's evaluate our models performance on the test set

In [23]:
score = model.evaluate(X_test, y_test, verbose=0)
print('loss=', score[0])
print('accuracy=', score[1])

loss= 0.680687665939331
accuracy= 0.5652680397033691


With this first model iteration, we are seeing similar training accuracy to test set accuracy; at 55%, it is slightly better than a random guess, and nothing to call home about. However, this model serves as a good foundational neural network to build upon with more sophisticated feature engineering, a more complex neural network architecture, and cross-validation. 